# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [8]:
# TODO
df, df.shape, df.dtypes, df.isnull().sum(), df.duplicated().sum()


(   order_id           item   category  qty   price                   ts
 0         1   Cheeseburger       Food  2.0   $7.50  2026-09-05T12:03:00
 1         1   Cheeseburger       Food  2.0   $7.50  2026-09-05T12:03:00
 2         2  cheese burger       food  1.0     7.5     09/05/2026 12:40
 3         3    Foam Finger      Merch  NaN      12  2026-09-05 13:00:00
 4         4   UVA T-Shirt     Apparel  2.0  $24.00     2026-09-05 13:05
 5         5    Rain Poncho   RainGear -3.0       6  2026-09-05T13:20:00
 6         6    rain poncho  rain-gear  4.0   $6.00                  NaN
 7         7            NaN      Merch  1.0      12  2026-09-05T14:00:00,
 (8, 6),
 order_id      int64
 item         object
 category     object
 qty         float64
 price        object
 ts           object
 dtype: object,
 order_id    0
 item        1
 category    0
 qty         1
 price       0
 ts          1
 dtype: int64,
 np.int64(1))

**What is wrong with this data?** List at least five specific problems:

1.  There's a duplicate row
2.  There are missing values in 'item', 'qty', and 'ts'
3.  'price' is a string, but should be a float
4.  There is inconsistent formatting. For example, the same item, 'Cheeseburger ', is spelled 'cheese burger', and 'RainGear' is also spelled 'rain-gear'.
5.  'qty' contains a negative value (not possible)

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [9]:
removed = df.duplicated().sum() # how many duplicates were there?
clean = df.drop_duplicates().copy()  # df with duplicates dropped, copied

log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [13]:
clean['price'] = clean['price'].astype(str).str.strip('$').astype(float)
assert clean['price'].dtype == float

log('price', 'coerced to float', len(clean))
assert len(clean) == raw_rows - removed

[price] coerced to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [19]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isnull().sum()    # count of NaN quantities
negative = (clean['qty'] < 0).sum()   # count of negative quantities

clean = clean[clean['qty'] >= 0].copy() # Drop rows with negative quantities
clean['qty'] = clean['qty'].fillna(0) # Fill NaN quantities with 0

log('qty', 'filled NaN with 0', missing)
log('qty', 'dropped negative quantities', negative)

[qty] filled NaN with 0 (1 row(s))
[qty] dropped negative quantities (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [21]:
print('before:', sorted(clean['category'].unique()))

initial_distinct_categories = len(clean['category'].unique())

# lowercase, strip, remove punctuation
clean['category'] = clean['category'].str.lower().str.strip()
clean['category'] = clean['category'].str.replace(r'[^a-z0-9\s]', '', regex=True) # remove punctuation

# CATEGORY_MAP for the judgment calls
CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Apparel',
    'raingear': 'Rain Gear',
    'rain-gear': 'Rain Gear'
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)

final_distinct_categories = len(clean['category'].unique())

print('after: ', sorted(clean['category'].unique()))
log('category', f'normalized case and mapped variants, {initial_distinct_categories} -> {final_distinct_categories} distinct categories', initial_distinct_categories - final_distinct_categories)

before: ['Apparel', 'Food', 'Merch', 'Rain Gear']
after:  ['Apparel', 'Food', 'Merch', 'rain gear']
[category] normalized case and mapped variants, 4 -> 4 distinct categories (0 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [25]:
print('before:', sorted([str(x) for x in clean['item'].unique()]))

initial_distinct_items = len(clean['item'].unique())

# lowercase, strip, remove punctuation
clean['item'] = clean['item'].str.lower().str.strip()
clean['item'] = clean['item'].str.replace(r'[^a-z0-9\s]', '', regex=True)

# Map variant spellings
ITEM_MAP = {
    'cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'uva tshirt': 'UVA T-Shirt',
    'rain poncho': 'Rain Poncho'
}
clean['item'] = clean['item'].replace(ITEM_MAP)

# Handle missing item names (NaNs)
missing_items = clean['item'].isnull().sum()
if missing_items > 0:
    clean['item'] = clean['item'].fillna('Unknown')
    log('item', 'filled NaN with Unknown', missing_items)

final_distinct_items = len(clean['item'].unique())

print('after: ', sorted(clean['item'].unique()))
log('item', f'normalized case and mapped variants, {initial_distinct_items} -> {final_distinct_items} distinct items', initial_distinct_items - final_distinct_items)

before: ['Cheeseburger', 'UVA T-Shirt ', 'cheese burger', 'nan', 'rain poncho']
[item] filled NaN with Unknown (1 row(s))
after:  ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt', 'Unknown']
[item] normalized case and mapped variants, 5 -> 4 distinct items (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [26]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')

failed_conversions = clean['ts'].isnull().sum()
log('ts', 'coerced non-datetime to NaT', failed_conversions)

clean['hour'] = clean['ts'].dt.hour
log('ts', 'extracted hour from timestamp', clean['hour'].count())

[ts] coerced non-datetime to NaT (3 row(s))
[ts] extracted hour from timestamp (2 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [27]:
# Assertions to catch regressions
assert clean.duplicated().sum() == 0, "There are still duplicate rows after cleaning!"
assert clean['price'].dtype == float, "Price column is not float type!"
assert (clean['qty'] >= 0).all(), "Quantity column contains negative values!"
assert clean['category'].isnull().sum() == 0, "Category column contains missing values!"
assert clean['item'].isnull().sum() == 0, "Item column contains missing values!"
assert pd.api.types.is_datetime64_any_dtype(clean['ts']), "Timestamp column is not datetime type!"

# Compute revenue
clean['revenue'] = clean['qty'] * clean['price']

# Print totals
print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

rows: 5
units: 10.0
revenue: 106.5
distinct categories: 4


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [ ]:
show_log()

**The decision that mattered most:** Dropping rows with negative quantities. This decision removed a row that would have contributed -18.0 to the revenue, effectively increasing the total revenue.

**Revenue with it:** 106.5  **Revenue without it:** 88.5

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [28]:
rows_after = 5
revenue_after = 106.5
biggest_decision = 'dropping negative quantities'
revenue_other_way = 88.5

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 106.5
decision that mattered: dropping negative quantities
revenue the other way: 88.5
